In [1]:
"""
DATA CHALLENGE — DIAMETER
Quality audit and characterization of DBH data for Plot 8 (eucalyptus inventory)
 
Course: Forest Mensuration
Author: [your name / group]
 
This script answers the 6 questions of the challenge:
  1. Descriptive statistics of diameter, per subplot and for the whole stand
  2. Diameter classes (Sturges' rule) + frequency tables + estimate of
     trees with DBH >= 15 cm across the 48.7 ha field
  3. Basal area (m2) and basal area per hectare (m2/ha), per subplot and total
  4. Data quality audit (outliers via IQR)
  5. DBH histograms (per subplot and for the whole stand) + interpretation
  6. Estimate of the area of each subplot and trees/ha, based on the
     3x3 m spacing
"""
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
 
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
 
OUT_DIR = r"C:\Users\DANIYAL HUSSAIN\Desktop\2 semester (2026)\semester 2 (2026)\Forest Management (Cristian)\Diameter"
os.makedirs(OUT_DIR, exist_ok=True)
 
# -----------------------------------------------------------------------
# 0. DATA LOADING AND PREPARATION
# -----------------------------------------------------------------------
# Instead of hardcoding the exact filename (accented characters like
# "inventário" are a common source of FileNotFoundError on Windows),
# we auto-detect the first .xlsx file sitting in OUT_DIR.
xlsx_candidates = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".xlsx")]
 
if len(xlsx_candidates) == 0:
    raise FileNotFoundError(
        f"No .xlsx file found in:\n{OUT_DIR}\n"
        "Move/copy your inventory spreadsheet into this folder and re-run."
    )
elif len(xlsx_candidates) > 1:
    print(f"WARNING: multiple .xlsx files found, using the first one: {xlsx_candidates}")
 
INPUT_FILE = os.path.join(OUT_DIR, xlsx_candidates[0])
print(f"Loading input file: {INPUT_FILE}")
 
df = pd.read_excel(INPUT_FILE)
 
# Standardize column names to English for clean, readable code
df = df.rename(columns={
    'FILIAL': 'branch',
    'IDADE': 'age',
    'REGIME': 'regime',
    'ESPAÇAMENTO': 'spacing',
    'CLONE': 'clone',
    'MÊS DE PLANTIO': 'planting_month',
    'AI': 'ai',
    'TALHÃO': 'stand',
    'SÍTIO': 'site',
    'PARCELA': 'plot',
    'FILEIRA': 'row',
    'ÁRVORE': 'tree',
    'DAP': 'dbh',
    'HT': 'ht',
})
 
print("=" * 70)
print("Dataset dimensions:", df.shape)
print("Sampled plots:", sorted(df['plot'].unique()))
print("Trees per plot:")
print(df.groupby('plot').size())
print("=" * 70)

Loading input file: C:\Users\DANIYAL HUSSAIN\Desktop\2 semester (2026)\semester 2 (2026)\Forest Management (Cristian)\Diameter\inventário.xlsx
Dataset dimensions: (215, 14)
Sampled plots: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Trees per plot:
plot
1    37
2    36
3    37
4    41
5    32
6    32
dtype: int64


c:\Users\DANIYAL HUSSAIN\miniconda3\envs\data_analysis\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [2]:
# -----------------------------------------------------------------------
# QUESTION 6 (solved first since plot area is used in Q3)
# Estimate of the area of each plot and the number of trees/ha
# -----------------------------------------------------------------------
#
# ASSUMPTION ADOPTED (also documented in the README):
# The spacing is 3 x 3 m. Each plot has 7 rows (row = 1..7).
# The maximum number of trees observed in any single row, across all
# plots, is 7 -> this indicates that the NOMINAL design of the plot is
# a square grid of 7 x 7 = 49 planting positions (7 rows x 7 positions
# per row), and that the observed counts (< 49 trees) reflect
# gaps/replanting mortality, not a smaller plot design.
#
# Nominal plot area = (7 rows x 3 m) x (7 positions x 3 m)
#                    = 21 m x 21 m = 441 m2 = 0.0441 ha
#
# This is the same nominal area for all 6 plots (same sampling design).
# The REAL trees/ha of each plot is then calculated by dividing the
# number of trees actually measured (survivors) by the plot's nominal
# area.

max_trees_per_row = df.groupby(['plot', 'row']).size().max()
print(f"\n[Q6] Maximum number of trees observed in a single row: {max_trees_per_row}")

n_rows = 7
n_positions_per_row = 7  # deduced from the maximum observed per row
spacing = 3.0  # m

plot_area_m2 = (n_rows * spacing) * (n_positions_per_row * spacing)
plot_area_ha = plot_area_m2 / 10_000

print(f"[Q6] Nominal area per plot: {plot_area_m2:.0f} m2 = {plot_area_ha:.4f} ha")

trees_per_plot = df.groupby('plot').size().rename('n_trees')
q6 = trees_per_plot.to_frame()
q6['plot_area_m2'] = plot_area_m2
q6['plot_area_ha'] = plot_area_ha
q6['trees_per_ha'] = q6['n_trees'] / plot_area_ha

# nominal planting density (if there had been no mortality/gaps)
nominal_density_ha = 10_000 / (spacing * spacing)
print(f"[Q6] Nominal planting density (no gaps): {nominal_density_ha:.1f} trees/ha")
print("\n[Q6] Trees/ha per plot (accounting for observed gaps):")
print(q6)

# total sampled area and average observed density (used in Q2's expansion)
total_sampled_area_ha = plot_area_ha * df['plot'].nunique()
avg_observed_density_ha = df.shape[0] / total_sampled_area_ha
print(f"\n[Q6] Total sampled area (6 plots): {total_sampled_area_ha:.4f} ha")
print(f"[Q6] Average observed density (with gaps): {avg_observed_density_ha:.1f} trees/ha")

q6.to_csv(f"{OUT_DIR}/Q6_plot_area_density.csv")


[Q6] Maximum number of trees observed in a single row: 7
[Q6] Nominal area per plot: 441 m2 = 0.0441 ha
[Q6] Nominal planting density (no gaps): 1111.1 trees/ha

[Q6] Trees/ha per plot (accounting for observed gaps):
      n_trees  plot_area_m2  plot_area_ha  trees_per_ha
plot                                                   
1          37         441.0        0.0441    839.002268
2          36         441.0        0.0441    816.326531
3          37         441.0        0.0441    839.002268
4          41         441.0        0.0441    929.705215
5          32         441.0        0.0441    725.623583
6          32         441.0        0.0441    725.623583

[Q6] Total sampled area (6 plots): 0.2646 ha
[Q6] Average observed density (with gaps): 812.5 trees/ha


In [3]:
# -----------------------------------------------------------------------
# QUESTION 1 — Descriptive statistics of diameter
# -----------------------------------------------------------------------

def dbh_descriptives(series):
    d = series.values
    n = len(d)
    dm = d.mean()                              # arithmetic mean diameter
    dg = np.sqrt((d ** 2).mean())              # quadratic mean diameter
    sd = d.std(ddof=1)                          # sample standard deviation
    dmin = d.min()
    dmax = d.max()
    cv = (sd / dm) * 100                        # coefficient of variation (%)
    return pd.Series({
        'n_trees': n,
        'arithmetic_mean_dbh_cm': dm,
        'quadratic_mean_dbh_cm': dg,
        'standard_deviation_cm': sd,
        'min_dbh_cm': dmin,
        'max_dbh_cm': dmax,
        'cv_%': cv,
    })

q1_per_plot = df.groupby('plot')['dbh'].apply(dbh_descriptives).unstack()
q1_stand = dbh_descriptives(df['dbh']).to_frame('STAND (overall)').T

q1 = pd.concat([q1_per_plot, q1_stand])
q1.index.name = 'plot'

print("\n" + "=" * 70)
print("[Q1] Descriptive statistics of DBH per plot and for the whole stand")
print("=" * 70)
print(q1.round(3))

q1.round(3).to_csv(f"{OUT_DIR}/Q1_descriptive_statistics.csv")



[Q1] Descriptive statistics of DBH per plot and for the whole stand
                 n_trees  arithmetic_mean_dbh_cm  quadratic_mean_dbh_cm  standard_deviation_cm  min_dbh_cm  \
plot                                                                                                         
1                   37.0                  15.805                 16.101                  3.112       7.946   
2                   36.0                  15.637                 15.943                  3.154       7.849   
3                   37.0                  14.954                 15.169                  2.579       8.749   
4                   41.0                  15.588                 15.757                  2.328       8.450   
5                   32.0                  17.200                 17.484                  3.187       8.399   
6                   32.0                  15.943                 16.249                  3.191       5.050   
STAND (overall)    215.0                  15.817   

In [4]:
# -----------------------------------------------------------------------
# QUESTION 2 — Diameter classes (Sturges) + frequencies
#              + estimate of trees with DBH >= 15 cm across 48.7 ha
# -----------------------------------------------------------------------

def sturges_classes(series, label):
    """Builds a frequency distribution table using Sturges' rule."""
    d = series.values
    n = len(d)
    k = math.ceil(1 + 3.322 * math.log10(n))          # number of classes (Sturges)
    amplitude = d.max() - d.min()
    h = amplitude / k                                    # class width

    # class boundaries
    limits = np.array([d.min() + i * h for i in range(k + 1)])
    limits[-1] += 1e-6  # ensures the max value falls into the last class

    freq_abs, edges = np.histogram(d, bins=limits)
    freq_rel = freq_abs / n * 100
    freq_cum = np.cumsum(freq_abs)
    freq_cum_rel = np.cumsum(freq_rel)

    table = pd.DataFrame({
        'class': [f"{edges[i]:.2f} |- {edges[i+1]:.2f}" for i in range(k)],
        'midpoint_cm': [(edges[i] + edges[i+1]) / 2 for i in range(k)],
        'abs_frequency': freq_abs,
        'rel_frequency_%': freq_rel.round(2),
        'cum_frequency': freq_cum,
        'cum_frequency_%': freq_cum_rel.round(2),
    })
    print(f"\n--- Diameter classes ({label}) | n={n}, k={k} classes, h={h:.2f} cm ---")
    print(table.to_string(index=False))
    return table, k, h

print("\n" + "=" * 70)
print("[Q2] Frequency distribution by diameter class (Sturges' rule)")
print("=" * 70)

class_tables = {}
with pd.ExcelWriter(f"{OUT_DIR}/Q2_diameter_classes.xlsx") as writer:
    for p in sorted(df['plot'].unique()):
        sub = df.loc[df['plot'] == p, 'dbh']
        table, k, h = sturges_classes(sub, f"Plot {p}")
        class_tables[p] = table
        table.to_excel(writer, sheet_name=f"Plot_{p}", index=False)

    overall_table, k_overall, h_overall = sturges_classes(df['dbh'], "STAND (all plots)")
    overall_table.to_excel(writer, sheet_name="Stand_overall", index=False)

# --- Estimate of trees with DBH >= 15 cm in the 48.7 ha field ---
FIELD_AREA_HA = 48.7

n_dbh_ge_15 = (df['dbh'] >= 15).sum()
prop_dbh_ge_15 = n_dbh_ge_15 / len(df)

expansion_factor = FIELD_AREA_HA / total_sampled_area_ha
estimated_trees_ge_15 = n_dbh_ge_15 * expansion_factor
estimated_total_trees = len(df) * expansion_factor

print(f"\n[Q2] Sampled trees with DBH >= 15 cm: {n_dbh_ge_15} of {len(df)} "
      f"({prop_dbh_ge_15*100:.1f}%)")
print(f"[Q2] Sampled area: {total_sampled_area_ha:.4f} ha | Field area: {FIELD_AREA_HA} ha")
print(f"[Q2] Expansion factor (field/sample): {expansion_factor:.2f}x")
print(f"[Q2] Estimated total trees in the field: {estimated_total_trees:,.0f}")
print(f"[Q2] Estimated trees with DBH >= 15 cm in the field: {estimated_trees_ge_15:,.0f}")



[Q2] Frequency distribution by diameter class (Sturges' rule)

--- Diameter classes (Plot 1) | n=37, k=7 classes, h=1.82 cm ---
         class  midpoint_cm  abs_frequency  rel_frequency_%  cum_frequency  cum_frequency_%
  7.95 |- 9.77     8.856994              2             5.41              2             5.41
 9.77 |- 11.59    10.678847              1             2.70              3             8.11
11.59 |- 13.41    12.500699              5            13.51              8            21.62
13.41 |- 15.23    14.322551              6            16.22             14            37.84
15.23 |- 17.06    16.144403             10            27.03             24            64.86
17.06 |- 18.88    17.966255              6            16.22             30            81.08
18.88 |- 20.70    19.788108              7            18.92             37           100.00

--- Diameter classes (Plot 2) | n=36, k=7 classes, h=1.71 cm ---
         class  midpoint_cm  abs_frequency  rel_frequency_%  cum_freq

In [5]:
# -----------------------------------------------------------------------
# QUESTION 4 — Data quality audit / outliers (criterion: IQR)
# (done before Q3 to decide whether suspicious values enter the basal
#  area calculation)
# -----------------------------------------------------------------------

def flag_outliers_iqr(series, k=1.5):
    q1v = series.quantile(0.25)
    q3v = series.quantile(0.75)
    iqr = q3v - q1v
    lower = q1v - k * iqr
    upper = q3v + k * iqr
    return lower, upper

print("\n" + "=" * 70)
print("[Q4] Data quality audit — Interquartile Range (IQR) criterion")
print("=" * 70)

outlier_rows = []
for p in sorted(df['plot'].unique()):
    sub = df[df['plot'] == p]
    lower, upper = flag_outliers_iqr(sub['dbh'])
    suspicious = sub[(sub['dbh'] < lower) | (sub['dbh'] > upper)]
    print(f"Plot {p}: IQR limits = [{lower:.2f}, {upper:.2f}] cm "
          f"-> {len(suspicious)} suspicious value(s)")
    if len(suspicious) > 0:
        print(suspicious[['plot', 'row', 'tree', 'dbh']].to_string(index=False))
        outlier_rows.append(suspicious.assign(lower_limit=lower, upper_limit=upper))

if outlier_rows:
    outliers_df = pd.concat(outlier_rows)
    outliers_df.to_csv(f"{OUT_DIR}/Q4_suspicious_values.csv", index=False)
    print(f"\n[Q4] Total suspicious values (all plots): {len(outliers_df)}")
else:
    outliers_df = pd.DataFrame()
    print("\n[Q4] No suspicious values found (IQR criterion, k=1.5) in any plot.")

# criterion also applied to the stand as a whole, for reference
lower_overall, upper_overall = flag_outliers_iqr(df['dbh'])
susp_overall = df[(df['dbh'] < lower_overall) | (df['dbh'] > upper_overall)]
print(f"\n[Q4] Reference (whole stand): IQR limits = "
      f"[{lower_overall:.2f}, {upper_overall:.2f}] cm -> "
      f"{len(susp_overall)} value(s) outside the range")

# None of the flagged suspicious values were removed from the basal area
# calculation below (see README for the rationale: values are within the
# biologically plausible range for the species/age).


[Q4] Data quality audit — Interquartile Range (IQR) criterion
Plot 1: IQR limits = [7.20, 24.54] cm -> 0 suspicious value(s)
Plot 2: IQR limits = [5.87, 25.85] cm -> 0 suspicious value(s)
Plot 3: IQR limits = [8.10, 21.84] cm -> 0 suspicious value(s)
Plot 4: IQR limits = [9.15, 22.48] cm -> 1 suspicious value(s)
 plot  row  tree      dbh
    4    2     8 8.449852
Plot 5: IQR limits = [8.96, 25.70] cm -> 1 suspicious value(s)
 plot  row  tree      dbh
    5    2     3 8.399405
Plot 6: IQR limits = [9.95, 22.68] cm -> 2 suspicious value(s)
 plot  row  tree      dbh
    6    5    22 9.648834
    6    6    29 5.049752

[Q4] Total suspicious values (all plots): 4

[Q4] Reference (whole stand): IQR limits = [7.72, 23.92] cm -> 1 value(s) outside the range


In [6]:
# -----------------------------------------------------------------------
# QUESTION 3 — Basal area (m2) and basal area per hectare (m2/ha)
# -----------------------------------------------------------------------

df['ba_m2'] = (math.pi / 4) * (df['dbh'] / 100) ** 2  # individual cross-sectional area (m2)

ba_per_plot = df.groupby('plot')['ba_m2'].sum().rename('basal_area_m2')
q3 = ba_per_plot.to_frame()
q3['plot_area_ha'] = plot_area_ha
q3['basal_area_m2_ha'] = q3['basal_area_m2'] / q3['plot_area_ha']

ba_total = df['ba_m2'].sum()
ba_total_ha = ba_total / total_sampled_area_ha

print("\n" + "=" * 70)
print("[Q3] Basal area per plot")
print("=" * 70)
print(q3.round(4))
print(f"\n[Q3] Total basal area (6 plots, {total_sampled_area_ha:.4f} ha sampled): "
      f"{ba_total:.4f} m2")
print(f"[Q3] Average basal area per hectare (stand): {ba_total_ha:.3f} m2/ha")

q3.round(4).to_csv(f"{OUT_DIR}/Q3_basal_area.csv")


[Q3] Basal area per plot
      basal_area_m2  plot_area_ha  basal_area_m2_ha
plot                                               
1            0.7533        0.0441           17.0824
2            0.7187        0.0441           16.2967
3            0.6687        0.0441           15.1623
4            0.7995        0.0441           18.1297
5            0.7683        0.0441           17.4216
6            0.6636        0.0441           15.0476

[Q3] Total basal area (6 plots, 0.2646 ha sampled): 4.3721 m2
[Q3] Average basal area per hectare (stand): 16.523 m2/ha


In [7]:
# -----------------------------------------------------------------------
# QUESTION 5 — DBH histograms (per plot and for the whole stand)
# -----------------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, p in enumerate(sorted(df['plot'].unique())):
    sub = df.loc[df['plot'] == p, 'dbh']
    axes[i].hist(sub, bins='sturges', color='#4C7C4C', edgecolor='black', alpha=0.85)
    axes[i].set_title(f"Plot {p} (n={len(sub)})")
    axes[i].set_xlabel("DBH (cm)")
    axes[i].set_ylabel("Frequency")
fig.suptitle("DBH distribution per plot — Stand 8 (Eucalyptus)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT_DIR}/Q5_histogram_per_plot.png", dpi=150)
plt.close(fig)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.hist(df['dbh'], bins='sturges', color='#2E5A88', edgecolor='black', alpha=0.85)
ax2.set_title(f"DBH distribution — Full stand 8 (n={len(df)})")
ax2.set_xlabel("DBH (cm)")
ax2.set_ylabel("Frequency")
ax2.axvline(df['dbh'].mean(), color='red', linestyle='--', label=f"Mean = {df['dbh'].mean():.1f} cm")
ax2.legend()
fig2.tight_layout()
fig2.savefig(f"{OUT_DIR}/Q5_histogram_stand_overall.png", dpi=150)
plt.close(fig2)

print("\n" + "=" * 70)
print("[Q5] Histograms saved to:")
print(f"  - {OUT_DIR}/Q5_histogram_per_plot.png")
print(f"  - {OUT_DIR}/Q5_histogram_stand_overall.png")
print("=" * 70)

print("\nSCRIPT FINISHED. All outputs were written to:", OUT_DIR)


[Q5] Histograms saved to:
  - C:\Users\DANIYAL HUSSAIN\Desktop\2 semester (2026)\semester 2 (2026)\Forest Management (Cristian)\Diameter/Q5_histogram_per_plot.png
  - C:\Users\DANIYAL HUSSAIN\Desktop\2 semester (2026)\semester 2 (2026)\Forest Management (Cristian)\Diameter/Q5_histogram_stand_overall.png

SCRIPT FINISHED. All outputs were written to: C:\Users\DANIYAL HUSSAIN\Desktop\2 semester (2026)\semester 2 (2026)\Forest Management (Cristian)\Diameter
